[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week5_nlp_llms/day39_building_with_apis/day39_notebook.ipynb)

# Day 39 / 42: Building with APIs (OpenAI, Gemini, Claude)
### #42DaysOfML | Week 5: NLP and LLMs

**Resources used to build this notebook:**
- [OpenAI API documentation](https://platform.openai.com/docs)
- [Anthropic API documentation](https://docs.anthropic.com)
- [Google Gemini API documentation](https://ai.google.dev/docs)
- [mlabonne/llm-course](https://github.com/mlabonne/llm-course) — LLM engineer track reference

---

## What You'll Learn
1. Token counting before sending a request — avoid surprises
2. Real cost math across OpenAI, Anthropic, and Gemini
3. Retry logic with exponential backoff — handle rate limits correctly
4. Streaming responses — why it matters for user experience
5. Structured outputs (JSON mode) — reliable parsing without regex
6. Semantic caching — skip LLM calls for similar queries
7. System prompts, roles, and conversation history
8. Build a complete resume screener using structured output

---

In [ ]:
!pip install openai anthropic google-generativeai tiktoken sentence-transformers matplotlib numpy tenacity --quiet

## The Concept

Calling an LLM API looks simple: send text, receive text. But in production, four problems appear quickly:

1. **Cost surprises:** You don't know how much a call will cost until after you make it — unless you count tokens first. A prompt that seems short can balloon with a system prompt and few-shot examples.

2. **Rate limit failures:** Every API has request-per-minute and token-per-minute limits. Hitting them throws errors. Without retry logic, your application crashes.

3. **Unparseable output:** Ask an LLM to return JSON and it returns markdown-wrapped JSON, or adds an explanation before the JSON, or uses single quotes instead of double. Without structured output, you spend hours writing regex parsers that break on edge cases.

4. **Redundant spend:** The same or similar question gets asked repeatedly. Without caching, every request costs money even when the answer is already known.

This notebook solves all four problems with production-tested patterns, then applies them to a real mini-app: a resume screener that reads a job description and a resume, then returns a structured evaluation with a fit score, strengths, gaps, and a hiring recommendation.

In [ ]:
import tiktoken
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------------------------------
# Section 1: Token Counting Before Sending
# Always count tokens BEFORE the API call, not after
# ----------------------------------------------------------------

def count_tokens(text: str, model: str = "gpt-3.5-turbo") -> int:
    """
    Count tokens using tiktoken — same tokenizer OpenAI uses internally.
    cl100k_base: used by GPT-3.5-turbo, GPT-4, GPT-4o
    """
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))


def count_messages_tokens(messages: list, model: str = "gpt-3.5-turbo") -> int:
    """
    Count tokens for a full messages list (system + user + assistant).
    Each message adds 4 tokens of overhead for role/formatting.
    """
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    total = 3  # reply priming
    for msg in messages:
        total += 4  # per-message overhead
        for val in msg.values():
            total += len(enc.encode(str(val)))
    return total


# Token count examples
texts = [
    ("Hello",                                                       "Single word"),
    ("The transformer architecture was introduced in 2017.",         "One sentence"),
    ("You are an expert ML engineer. " * 10,                         "Repeated system prompt (x10)"),
    ("Analyze this resume and evaluate fitness for the ML engineer role. " +
     "Consider technical skills, experience level, and project quality. " * 20, "Detailed prompt"),
]

print("TOKEN COUNTS (GPT-3.5-turbo / cl100k_base tokenizer)")
print("=" * 60)
for text, label in texts:
    n = count_tokens(text)
    cost_input  = n / 1000 * 0.0015
    cost_10k    = cost_input * 10_000
    print(f"  {label}")
    print(f"    Tokens: {n:,}  |  Cost/call: ${cost_input:.5f}  |  Cost/10K calls: ${cost_10k:.2f}")
    print()

# Conversation token counting
messages = [
    {"role": "system",    "content": "You are an expert ML engineer. Answer concisely."},
    {"role": "user",      "content": "What is the difference between RAG and fine-tuning?"},
    {"role": "assistant", "content": "RAG retrieves context at inference time. Fine-tuning updates model weights."},
    {"role": "user",      "content": "When should I use each?"},
]
total = count_messages_tokens(messages)
print(f"Full conversation ({len(messages)} messages): {total} tokens")
print(f"  Context window usage: {total/128000*100:.2f}% of GPT-4 128K limit")

In [ ]:
# ----------------------------------------------------------------
# Section 2: Cost Comparison Across APIs
# Prices as of mid-2024. Always check official pricing pages.
# ----------------------------------------------------------------

PRICING = {
    "gpt-3.5-turbo":     {"input": 0.0015, "output": 0.002,   "provider": "OpenAI"},
    "gpt-4o":            {"input": 0.005,  "output": 0.015,   "provider": "OpenAI"},
    "gpt-4":             {"input": 0.03,   "output": 0.06,    "provider": "OpenAI"},
    "claude-3-haiku":    {"input": 0.00025,"output": 0.00125, "provider": "Anthropic"},
    "claude-3-sonnet":   {"input": 0.003,  "output": 0.015,   "provider": "Anthropic"},
    "claude-3-opus":     {"input": 0.015,  "output": 0.075,   "provider": "Anthropic"},
    "gemini-1.5-flash":  {"input": 0.00035,"output": 0.00105, "provider": "Google"},
    "gemini-1.5-pro":    {"input": 0.0035, "output": 0.0105,  "provider": "Google"},
}

def estimate_cost(prompt: str, model: str, estimated_output_tokens: int = 200) -> dict:
    p = PRICING[model]
    in_tokens  = count_tokens(prompt, "gpt-3.5-turbo")  # approximate for all models
    cost = (in_tokens/1000)*p["input"] + (estimated_output_tokens/1000)*p["output"]
    return {
        "model":          model,
        "provider":       p["provider"],
        "input_tokens":   in_tokens,
        "output_tokens":  estimated_output_tokens,
        "cost_per_call":  round(cost, 6),
        "cost_per_10k":   round(cost * 10_000, 2),
        "cost_per_100k":  round(cost * 100_000, 2),
    }

sample_prompt = """You are an expert ML engineer.
Analyze this resume for a Senior ML Engineer position.
Return a structured evaluation with fit score, strengths, gaps, and recommendation.
Resume: [500 word resume content here]"""

print("COST COMPARISON — Resume Screener Prompt")
print("=" * 75)
print(f"{'Model':22s} {'Provider':12s} {'In tok':>8} {'$/call':>10} {'$/10K':>10} {'$/100K':>10}")
print("-" * 75)

all_costs = []
for model in PRICING:
    c = estimate_cost(sample_prompt, model, estimated_output_tokens=300)
    all_costs.append(c)
    print(f"{c['model']:22s} {c['provider']:12s} {c['input_tokens']:>8,} "
          f"{c['cost_per_call']:>10.5f} {c['cost_per_10k']:>10.2f} {c['cost_per_100k']:>10.2f}")

print(f"\nCheapest: {min(all_costs, key=lambda x: x['cost_per_call'])['model']}")
print(f"Most expensive: {max(all_costs, key=lambda x: x['cost_per_call'])['model']}")
print(f"Price range: {max(c['cost_per_call'] for c in all_costs) / min(c['cost_per_call'] for c in all_costs):.0f}x difference between cheapest and most expensive")

In [ ]:
# Cost visualisation
models_plot = [c['model'].replace('claude-3-','c3-').replace('gemini-1.5-','gem-') for c in all_costs]
costs_10k   = [c['cost_per_10k'] for c in all_costs]
provider_colors = {'OpenAI': '#2196F3', 'Anthropic': '#FF9800', 'Google': '#4CAF50'}
bar_colors = [provider_colors[c['provider']] for c in all_costs]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(models_plot, costs_10k, color=bar_colors, alpha=0.85, edgecolor='black')
for bar, c in zip(bars, costs_10k):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'${c:.0f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_ylabel('Cost per 10,000 calls ($)', fontsize=11)
axes[0].set_title('LLM API Cost per 10K Calls\nBlue=OpenAI, Orange=Anthropic, Green=Google', fontsize=11, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Caching impact
monthly_calls = np.arange(0, 100001, 10000)
cost_no_cache   = monthly_calls * all_costs[0]['cost_per_call']  # gpt-3.5-turbo
cost_with_cache = monthly_calls * all_costs[0]['cost_per_call'] * 0.35  # 65% cache hit
axes[1].plot(monthly_calls/1000, cost_no_cache,   'r-', linewidth=2.5, label='No cache')
axes[1].plot(monthly_calls/1000, cost_with_cache, 'g-', linewidth=2.5, label='65% cache hit rate')
axes[1].fill_between(monthly_calls/1000, cost_with_cache, cost_no_cache, alpha=0.15, color='green', label='Savings')
axes[1].set_xlabel('Monthly calls (thousands)', fontsize=11)
axes[1].set_ylabel('Monthly cost ($)', fontsize=11)
axes[1].set_title('Semantic Caching Impact\n(GPT-3.5-Turbo, 65% cache hit rate)', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout(); plt.show()

saving_at_100k = cost_no_cache[-1] - cost_with_cache[-1]
print(f"At 100K monthly calls with 65% cache hit rate:")
print(f"  Without cache: ${cost_no_cache[-1]:.2f}/month")
print(f"  With cache:    ${cost_with_cache[-1]:.2f}/month")
print(f"  Savings:       ${saving_at_100k:.2f}/month (${saving_at_100k*12:.0f}/year)")

In [ ]:
# ----------------------------------------------------------------
# Section 3: Retry Logic with Exponential Backoff
# ----------------------------------------------------------------
import time, random, os
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# Manual implementation — so you understand what tenacity does under the hood
def exponential_backoff(attempt: int, base: float = 1.0, max_wait: float = 60.0) -> float:
    """Wait = base * 2^attempt + random jitter. Caps at max_wait."""
    wait = min(base * (2 ** attempt) + random.uniform(0, 0.5), max_wait)
    return round(wait, 2)

print("EXPONENTIAL BACKOFF WAIT TIMES:")
print("=" * 40)
for attempt in range(6):
    wait = exponential_backoff(attempt)
    bar = '|' * int(wait)
    print(f"  Attempt {attempt}: wait {wait:5.1f}s  {bar}")

# Visualise
attempts = list(range(7))
waits = [exponential_backoff(a, base=1.0) for a in attempts]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(attempts, waits, 'r-o', linewidth=2.5, markersize=9)
for a, w in zip(attempts, waits):
    ax.text(a, w+0.5, f'{w:.1f}s', ha='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Attempt number', fontsize=12)
ax.set_ylabel('Wait before retry (seconds)', fontsize=12)
ax.set_title('Exponential Backoff: Wait doubles after each failed attempt\n'
             'Plus random jitter to avoid thundering herd problem', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


# Production retry wrapper using tenacity
def make_api_call_with_retry(client, messages, model="gpt-3.5-turbo", max_retries=4):
    """
    Retries on rate limits and transient errors.
    Uses tenacity for clean retry logic.
    """
    if client is None:
        print("[No API key — showing retry structure]")
        print(f"  Would call: {model}")
        print(f"  Max retries: {max_retries}")
        print(f"  Retry on: RateLimitError, APITimeoutError, InternalServerError")
        print(f"  Backoff: 1s -> 2s -> 4s -> 8s -> 16s (with jitter)")
        return None

    from openai import RateLimitError, APITimeoutError, InternalServerError

    @retry(
        stop=stop_after_attempt(max_retries),
        wait=wait_exponential(multiplier=1, min=1, max=60),
        retry=retry_if_exception_type((RateLimitError, APITimeoutError, InternalServerError))
    )
    def _call():
        return client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0,
            max_tokens=500,
            timeout=30
        )

    return _call()


print("\nRetry logic summary:")
print("  - Retry on: RateLimitError (429), APITimeoutError, InternalServerError (500)")
print("  - Do NOT retry on: AuthenticationError (wrong key), BadRequestError (bad prompt)")
print("  - Jitter prevents multiple clients hitting the API simultaneously after a limit")
print("  - tenacity.retry decorator is the production standard — clean and configurable")

In [ ]:
# ----------------------------------------------------------------
# Section 4: Structured Outputs — Reliable JSON from LLMs
# ----------------------------------------------------------------
import json, re

def extract_json(text: str):
    """
    Extract JSON from LLM output that may contain markdown fences,
    explanatory text, or other noise.
    """
    text = text.strip()
    # Remove markdown code fences
    if '```' in text:
        text = re.sub(r'```(?:json)?', '', text).replace('```', '').strip()
    # Find the first {...} block
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        text = match.group(0)
    try:
        return json.loads(text), None
    except json.JSONDecodeError as e:
        return None, str(e)


# Test various LLM output formats you'd actually see in production
test_outputs = [
    ('{"score": 0.85, "recommendation": "Strong hire", "gaps": ["MLOps experience"]}',
     "Clean JSON — ideal case"),

    ('```json\n{"score": 0.85, "recommendation": "Strong hire", "gaps": ["MLOps"]}\n```',
     "Markdown-wrapped JSON — common GPT-3.5 output"),

    ('Based on my analysis:\n{"score": 0.72, "recommendation": "Hire", "gaps": []}',
     "Preamble before JSON"),

    ("The candidate scores 85 out of 100. I recommend hiring them.",
     "No JSON — plain text answer"),

    ('{"score": 0.91, "recommendation": "Strong hire", "strengths": ["PhD", "5yr exp"]}',
     "Valid JSON with different keys"),
]

print("STRUCTURED OUTPUT EXTRACTION")
print("=" * 60)
for output, desc in test_outputs:
    obj, err = extract_json(output)
    status = 'PARSED' if obj else 'FAILED'
    print(f"  [{status}] {desc}")
    if obj:
        print(f"    Keys: {list(obj.keys())}")
    else:
        print(f"    Error: {err}")
    print()

print("Production best practice:")
print("  1. Use JSON mode (response_format={type: json_object}) for OpenAI models")
print("  2. Use Pydantic models with instructor library for schema validation")
print("  3. Always have a fallback extract_json() for models without native JSON mode")
print("  4. If parsing fails, log the raw output and retry with a more explicit prompt")

In [ ]:
# ----------------------------------------------------------------
# Section 5: Streaming Responses
# ----------------------------------------------------------------
import os, time

def stream_response(client, prompt: str, model: str = "gpt-3.5-turbo"):
    """
    Stream tokens as they're generated.
    With streaming: first token in ~300ms, user sees progress immediately.
    Without streaming: full response in ~2-5s, user sees blank screen.
    """
    if client is None:
        print("[No API key — simulating stream output]")
        simulated = "Streaming sends each token as soon as it's generated. "\
                    "The user sees output building in real time. "\
                    "This dramatically improves perceived latency."
        for word in simulated.split():
            print(word, end=' ', flush=True)
            time.sleep(0.05)
        print()
        return simulated

    full_response = ""
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        max_tokens=200
    )
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        print(delta, end='', flush=True)
        full_response += delta
    print()
    return full_response


client = None
if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()

print("STREAMING DEMO:")
print("=" * 50)
result = stream_response(client, "Explain what RAG is in two sentences.")

print("\n\nWhy streaming matters:")
print("  Non-streaming: user waits 3-5 seconds staring at a blank screen")
print("  Streaming:     user sees first word in <500ms, output builds in real time")
print("  ChatGPT, Claude, Gemini all stream by default — users expect it")
print("  For structured output (JSON): disable streaming so you can parse the full response")

In [ ]:
# ----------------------------------------------------------------
# Section 6: Calling All Three APIs — Unified Interface
# ----------------------------------------------------------------
import os

def call_openai(prompt: str, system: str = "", model: str = "gpt-3.5-turbo"):
    api_key = os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return f"[OpenAI] No API key. Would send: {prompt[:60]}..."
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(model=model, messages=messages,
                                          temperature=0, max_tokens=300)
    return resp.choices[0].message.content


def call_anthropic(prompt: str, system: str = "", model: str = "claude-3-haiku-20240307"):
    api_key = os.environ.get("ANTHROPIC_API_KEY", "")
    if not api_key:
        return f"[Anthropic] No API key. Would send: {prompt[:60]}..."
    import anthropic
    client = anthropic.Anthropic(api_key=api_key)
    resp = client.messages.create(
        model=model,
        max_tokens=300,
        system=system or "You are a helpful assistant.",
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.content[0].text


def call_gemini(prompt: str, system: str = "", model: str = "gemini-1.5-flash"):
    api_key = os.environ.get("GOOGLE_API_KEY", "")
    if not api_key:
        return f"[Gemini] No API key. Would send: {prompt[:60]}..."
    import google.generativeai as genai
    genai.configure(api_key=api_key)
    full_prompt = f"{system}\n\n{prompt}" if system else prompt
    resp = genai.GenerativeModel(model).generate_content(full_prompt)
    return resp.text


# Unified router
def call_llm(prompt: str, provider: str = "openai", system: str = "", **kwargs) -> str:
    """
    Unified interface for all three providers.
    In production: add fallback logic (if OpenAI fails, try Anthropic)
    """
    if provider == "openai":
        return call_openai(prompt, system, **kwargs)
    elif provider == "anthropic":
        return call_anthropic(prompt, system, **kwargs)
    elif provider == "gemini":
        return call_gemini(prompt, system, **kwargs)
    raise ValueError(f"Unknown provider: {provider}")


# Test all three with the same prompt
test_prompt = "In one sentence: what is the key difference between fine-tuning and RAG?"
system_msg  = "You are an ML expert. Be concise and precise."

print("MULTI-PROVIDER API TEST")
print("=" * 60)
for provider in ["openai", "anthropic", "gemini"]:
    result = call_llm(test_prompt, provider=provider, system=system_msg)
    print(f"\n[{provider.upper()}]")
    print(f"  {result[:150]}")

In [ ]:
# ----------------------------------------------------------------
# Section 7: Mini-App — Resume Screener with Structured Output
# Combines: system prompt, structured output, JSON parsing, cost tracking
# ----------------------------------------------------------------
import json

SCREENER_SYSTEM = """You are an expert technical recruiter specialising in Machine Learning roles.
Evaluate resumes objectively based on the job requirements.
Always return valid JSON matching the exact schema provided.
Do not include any text outside the JSON object."""

SCREENER_TEMPLATE = """Evaluate this candidate for the following role:

JOB DESCRIPTION:
{job_description}

CANDIDATE RESUME:
{resume}

Return ONLY this JSON schema (no other text):
{{
  "fit_score": <float 0.0-1.0>,
  "experience_match": <"junior"|"mid"|"senior"|"overqualified">,
  "strengths": [<list of 3 strings>],
  "gaps": [<list of up to 3 strings, empty if none>],
  "recommendation": <"strong_hire"|"hire"|"maybe"|"no_hire">,
  "reasoning": <string, 2 sentences max>
}}"""


def screen_resume(job_description: str, resume: str, client=None) -> dict:
    prompt = SCREENER_TEMPLATE.format(
        job_description=job_description,
        resume=resume
    )
    input_tokens = count_tokens(SCREENER_SYSTEM + prompt)
    print(f"Prompt tokens: {input_tokens:,}  |  Estimated cost: ${input_tokens/1000*0.0015:.4f}")

    if client:
        resp = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": SCREENER_SYSTEM},
                {"role": "user",   "content": prompt}
            ],
            temperature=0,
            max_tokens=400,
            response_format={"type": "json_object"}  # JSON mode guarantees valid JSON
        )
        raw = resp.choices[0].message.content
    else:
        # Simulated response for demonstration without API key
        raw = json.dumps({
            "fit_score": 0.82,
            "experience_match": "mid",
            "strengths": ["Strong Python and PyTorch skills", "Published ML research", "Production RAG experience"],
            "gaps": ["No MLOps/deployment experience", "No LLM fine-tuning experience"],
            "recommendation": "hire",
            "reasoning": "Strong ML fundamentals and relevant project experience. "
                         "Gaps in deployment and LLM fine-tuning can be addressed with onboarding."
        })
        print("[Using simulated response — add OPENAI_API_KEY for real evaluation]")

    result, err = extract_json(raw)
    if err:
        return {"error": err, "raw": raw}
    return result


# Test data
job_description = """
Senior ML Engineer — AI Products Team

Requirements:
- 4+ years of ML engineering experience
- Production experience with LLMs (RAG, fine-tuning, prompt engineering)
- Strong Python, PyTorch or TensorFlow
- MLOps: model deployment, monitoring, CI/CD for ML pipelines
- Experience with vector databases (FAISS, Pinecone, ChromaDB)

Nice to have:
- Published ML research or open-source contributions
- Experience with LLM fine-tuning (LoRA, QLoRA)
"""

resume = """
Priya Sharma — ML Engineer
3 years experience | priya@email.com

SKILLS: Python, PyTorch, scikit-learn, HuggingFace Transformers, FAISS, ChromaDB, LangChain

EXPERIENCE:
ML Engineer @ StartupAI (2022-present)
- Built a RAG pipeline serving 50K queries/day using FAISS + GPT-3.5
- Implemented semantic search over 500K documents with all-MiniLM-L6-v2
- Prompt engineering for 12 production use cases across 3 product teams

Data Scientist @ Infosys (2021-2022)
- Trained classification models (Random Forest, XGBoost) for churn prediction
- Built automated EDA pipeline reducing analysis time by 40%

EDUCATION: B.Tech Computer Science, IIT Bombay, 2021

RESEARCH: Co-authored paper on efficient RAG chunking strategies (arXiv 2023)
"""

print("RESUME SCREENER")
print("=" * 60)
result = screen_resume(job_description, resume, client=client)

print("\nEVALUATION RESULT:")
print("=" * 40)
if 'error' not in result:
    print(f"  Fit Score:          {result['fit_score']:.0%}")
    print(f"  Experience Match:   {result['experience_match']}")
    print(f"  Recommendation:     {result['recommendation'].upper()}")
    print(f"\n  Strengths:")
    for s in result.get('strengths', []):
        print(f"    + {s}")
    print(f"\n  Gaps:")
    for g in result.get('gaps', []):
        print(f"    - {g}")
    print(f"\n  Reasoning: {result.get('reasoning', '')}")
else:
    print(f"  Parse error: {result['error']}")

In [ ]:
# ----------------------------------------------------------------
# Section 8: Batch Resume Screening with Cost Tracking
# ----------------------------------------------------------------

resumes_batch = [
    ("Candidate A — Strong", resume),  # reuse Priya's resume
    ("Candidate B — Junior", """
    Alex Chen — ML Student
    1 year experience | alex@email.com
    SKILLS: Python, scikit-learn, NumPy, pandas
    EXPERIENCE: ML internship @ startup (6 months) - basic classification models
    EDUCATION: B.Sc. CS, 2023
    """),
    ("Candidate C — Senior", """
    Dr. Ravi Kumar — Principal ML Engineer
    8 years experience | ravi@email.com
    SKILLS: Python, PyTorch, TensorFlow, Kubernetes, MLflow, FAISS, Pinecone
    Fine-tuned LLaMA-2 and Mistral-7B using QLoRA for domain-specific tasks
    Built MLOps platform serving 5M predictions/day, 99.9% uptime
    EXPERIENCE: Staff ML Engineer @ BigTech (2019-present)
    EDUCATION: PhD Machine Learning, IIT Delhi, 2016
    """),
]

print("BATCH RESUME SCREENING")
print("=" * 60)
total_tokens = 0
batch_results = []

for name, res in resumes_batch:
    prompt = SCREENER_TEMPLATE.format(job_description=job_description, resume=res)
    tokens = count_tokens(SCREENER_SYSTEM + prompt)
    total_tokens += tokens
    result = screen_resume(job_description, res, client=None)  # simulated for demo
    batch_results.append((name, result, tokens))
    print(f"\n{name}: {result.get('fit_score',0):.0%} fit | {result.get('recommendation','N/A')} | {tokens} tokens")

total_cost = total_tokens / 1000 * 0.0015
print(f"\nBatch Summary:")
print(f"  Total tokens:  {total_tokens:,}")
print(f"  Total cost:    ${total_cost:.4f}")
print(f"  Cost per resume: ${total_cost/len(resumes_batch):.4f}")
print(f"\nAt 1,000 resumes/day:")
print(f"  Daily cost:   ${total_cost/len(resumes_batch)*1000:.2f}")
print(f"  Monthly cost: ${total_cost/len(resumes_batch)*1000*30:.2f}")
print(f"\nFor high volume: use claude-3-haiku (5x cheaper) with GPT-4 only for borderline cases.")

In [ ]:
# ----------------------------------------------------------------
# Section 9: Conversation History Management
# ----------------------------------------------------------------

class ConversationManager:
    """
    Manages multi-turn conversations with automatic context trimming.
    LLMs have no memory between API calls — you must send the full history.
    """

    def __init__(self, system: str, max_tokens: int = 3000):
        self.system   = system
        self.max_tokens = max_tokens
        self.history  = []

    def add_user(self, message: str):
        self.history.append({"role": "user", "content": message})
        self._trim_if_needed()

    def add_assistant(self, message: str):
        self.history.append({"role": "assistant", "content": message})

    def _trim_if_needed(self):
        """Remove oldest messages if approaching token limit."""
        messages = [{"role": "system", "content": self.system}] + self.history
        while count_messages_tokens(messages) > self.max_tokens and len(self.history) > 2:
            self.history.pop(0)  # remove oldest user message
            if self.history:
                self.history.pop(0)  # remove corresponding assistant message
            messages = [{"role": "system", "content": self.system}] + self.history

    def get_messages(self) -> list:
        return [{"role": "system", "content": self.system}] + self.history

    def token_count(self) -> int:
        return count_messages_tokens(self.get_messages())

    def chat(self, user_message: str, client=None) -> str:
        self.add_user(user_message)
        if client:
            resp = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=self.get_messages(),
                temperature=0.7,
                max_tokens=200
            )
            answer = resp.choices[0].message.content
        else:
            answer = f"[Simulated response to: '{user_message[:50]}']"
        self.add_assistant(answer)
        return answer


# Demo conversation
conv = ConversationManager(
    system="You are an expert ML engineer helping a student learn about LLMs.",
    max_tokens=3000
)

conversation_turns = [
    "What is the difference between RAG and fine-tuning?",
    "When would I use fine-tuning over RAG?",
    "What if I have very limited data — less than 100 examples?",
    "Can I combine both approaches?",
]

print("CONVERSATION WITH HISTORY MANAGEMENT")
print("=" * 55)
for turn in conversation_turns:
    response = conv.chat(turn, client=client)
    print(f"\nUser: {turn}")
    print(f"Assistant: {response[:120]}")
    print(f"[History: {len(conv.history)//2} turns | Tokens: {conv.token_count():,}]")

## Real World Problem: The $8,000 API Bill

A startup's API cost went from $800/month to $8,000/month between January and March without any significant increase in users.

**Root cause investigation revealed three problems:**

1. **No caching:** Their customer support bot answered the same 200 questions every day. Each answer triggered a full API call. 200 questions × 500 tokens × $0.003/1K × 30 days = $900/month just on repeated questions.

2. **Conversation history not trimmed:** Their multi-turn chatbot appended every message to history without checking token count. By message 15, each call was sending 8,000 tokens of context — most of it irrelevant to the current question.

3. **Wrong model for the task:** They used GPT-4 for every call, including simple classification tasks that GPT-3.5 handles equally well at 20x lower cost.

**Three-week fix:**
- Added semantic cache (Day 37 pattern): 65% hit rate on common questions → saved $585/month
- Added conversation trimming: kept only last 8 turns → cut average tokens per call by 60%
- Added model routing: simple tasks to GPT-3.5, complex reasoning to GPT-4 → 40% cost reduction on remaining calls

Combined effect: monthly bill dropped from $8,000 back to $1,100. Same product, same users, same quality.

## Interview Corner: MNC-Level Questions

---

**Q1: Your LLM application's monthly API cost doubled without an increase in users. What do you check first?**

*Answer direction:* Three things in order. Log average input token count per request over time — if it's growing, your prompts or conversation history are bloating. Check cache hit rate — if it dropped, you're re-answering questions you already answered. Check model distribution — if something changed routing more requests to a premium model, the cost multiplies immediately. Fix: token count logging on every request, cache hit rate as a dashboard metric, and model-per-request logging. These three metrics catch 90% of cost anomalies.

---

**Q2: Why does exponential backoff with jitter work better than fixed-interval retry?**

*Answer direction:* Fixed-interval retry causes the thundering herd problem. If 100 clients all hit a rate limit at the same time and all retry after exactly 1 second, they all retry simultaneously — causing another rate limit. Exponential backoff ensures each subsequent retry waits longer, giving the server time to recover. Jitter (random delay added to the wait) spreads retries across time even when multiple clients fail simultaneously. The combination means retry attempts naturally space out and the API recovers. AWS, Google, and every major API provider recommend exponential backoff with jitter as the standard retry pattern.

---

**Q3: What is the difference between temperature=0 and temperature=1 in a production API call, and when do you use each?**

*Answer direction:* Temperature controls the randomness of token sampling. At temperature=0, the model always picks the highest-probability next token — deterministic and reproducible. At temperature=1, sampling follows the raw probability distribution — more diverse but less predictable. Use temperature=0 for: structured output (JSON parsing fails on unexpected variations), factual QA (you want the most probable correct answer), classification tasks, evaluation pipelines (reproducibility matters). Use temperature=0.7-1.0 for: creative writing, brainstorming, generating multiple diverse options, chatbots where some variation feels natural. Rule: if you're parsing the output programmatically, use temperature=0.

---

**Q4: You need to build a chatbot that remembers the last 10 turns of conversation but stays within a 4,096 token context limit. How do you implement this?**

*Answer direction:* Sliding window with token-aware trimming. Maintain a list of message dicts (role + content). Before each API call, count tokens for system prompt + full history + new message. If over the limit, remove the oldest (user, assistant) pair and recount. Repeat until under the limit. Don't just count messages — count tokens, because message length varies. In practice: reserve 1,000 tokens for the output (max_tokens), which leaves 3,096 for input. Count tokens using tiktoken before the call, not after. This approach is what most production chatbots implement. LangChain's ConversationBufferWindowMemory and ConversationTokenBufferMemory implement exactly this.

---

**Q5: How do you implement a fallback strategy when your primary LLM API is down?**

*Answer direction:* Provider routing with health checking. Maintain a priority list: primary (e.g., GPT-4), secondary (e.g., Claude-3-Sonnet), tertiary (e.g., Gemini-1.5-Pro). On each call, try primary. On a 5xx error or timeout after 3 retries, immediately route to secondary. Log every fallback with timestamp, error type, and which provider served the request. If primary fails more than 5% of requests in a 5-minute window, automatically deprioritise it and alert on-call. Keep a compatibility layer that normalises API response formats across providers so the caller doesn't know which model served the request. LiteLLM is an open-source library that implements this exact pattern with a unified interface across 100+ LLM providers.

## ML Spotlight

**LiteLLM — One Interface for 100+ LLM APIs**

LiteLLM provides a single `completion()` function that works across OpenAI, Anthropic, Gemini, Cohere, Mistral, and 100+ other providers. You switch providers by changing a model string, not rewriting your code:

```python
from litellm import completion

# OpenAI
resp = completion(model="gpt-4", messages=[{"role": "user", "content": "Hello"}])

# Anthropic — same code, different model string
resp = completion(model="claude-3-sonnet-20240229", messages=[{"role": "user", "content": "Hello"}])

# Gemini — same code again
resp = completion(model="gemini/gemini-1.5-flash", messages=[{"role": "user", "content": "Hello"}])

# Built-in fallbacks
resp = completion(model="gpt-4", messages=[...], fallbacks=["claude-3-sonnet-20240229"])
```

It also provides built-in cost tracking, rate limit handling, and a proxy server for team-wide API key management.

GitHub: https://github.com/BerriAI/litellm

## Practice Exercise

**Task 1:** Add a `cost_tracker` to the `ConversationManager`. After each `.chat()` call, record the token count and estimated cost. Add a `.total_cost()` method that returns cumulative spend for the conversation.

**Task 2:** Extend the resume screener to handle batch screening with model routing:
- Use `claude-3-haiku` (cheapest) for initial screening
- Only escalate to `gpt-4` for candidates with `fit_score >= 0.75`
- Track and print total cost savings vs using GPT-4 for everything

**Task 3:** Implement a simple provider fallback:
```python
def call_with_fallback(prompt, providers=['openai','anthropic','gemini']):
    for provider in providers:
        try:
            result = call_llm(prompt, provider=provider)
            if result and not result.startswith('['):
                return result, provider
        except Exception as e:
            print(f"{provider} failed: {e}")
    return None, None
```
Test it by intentionally passing a wrong API key for the primary provider.

---

**Week 5 Complete.**

Week 5 Revision post coming next — 5 MNC interview questions, 3 practice projects combining Days 33-39, and a community challenge.

**Week 6 starts with Production:**
Day 41: Deploying an ML Model with FastAPI — from trained model to live REST API endpoint.